In [25]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [26]:
result_path = Path("../results/downstream_task")
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
method_name_replacer = {"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                            # "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "mrs-forest": "MRS", 
                       # "fw-mrs-temperature-svm": "FW-MRS-SVM", 
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-negative": "FW-MRS$_{Neg}$",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                          }
data_set_replacer = {"folktables_employment": "Employment", "folktables_income": "Income",
                               "breast_cancer": "Breast Cancer", "hr_analytics": "HR Analytic", "loan_prediction": "Loan",
                               "diabetes": "Diabetes", "german_credit": "German Credit", "bank_marketing": "Bank Marketing"}

In [27]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [28]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.870601,0.010491,0.826992,0.017307,less_positive_class,0.1,0.00,0.000000
1,KMM,folktables_employment,0.856631,0.013580,0.808776,0.021985,less_positive_class,0.1,0.00,0.000000
2,PSA,folktables_employment,0.867401,0.010937,0.823689,0.017363,less_positive_class,0.1,0.04,0.280000
3,MRS,folktables_employment,0.868702,0.010788,0.824747,0.017384,less_positive_class,0.1,354.00,74.799733
4,FW-MRS,folktables_employment,0.864302,0.011904,0.819250,0.018954,less_positive_class,0.1,397.50,54.490825
5,FW-MRS$_{Neg}$,folktables_employment,0.873639,0.010306,0.830388,0.016388,less_positive_class,0.1,350.10,77.591817
6,FW-MRS$_{SVM}$,folktables_employment,0.834438,0.013173,0.776956,0.023196,less_positive_class,0.1,271.80,31.044484
7,Uniform,folktables_income,0.838084,0.012878,0.788634,0.020309,less_positive_class,0.1,0.00,0.000000
8,KMM,folktables_income,0.820208,0.014266,0.765446,0.019706,less_positive_class,0.1,0.00,0.000000
9,PSA,folktables_income,0.831057,0.013985,0.780822,0.021689,less_positive_class,0.1,0.16,0.703136


In [29]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auroc_values = []
            std_auroc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
\\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.871\pm0.01$ & $0.857\pm0.01$ & $0.867\pm0.01$ & $0.869\pm0.01$ & $0.864\pm0.01$ & $0.874\pm0.01$ \\
Income & $0.838\pm0.01$ & $0.82\pm0.01$ & $0.831\pm0.01$ & $0.836\pm0.01$ & $0.834\pm0.01$ & $0.837\pm0.01$ \\
Breast Cancer & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.987\pm0.01$ & $0.987\pm0.01$ \\
HR Analytic & $0.753\pm0.02$ & $0.749\pm0.02$ & $0.75\pm0.02$ & $0.749\pm0.02$ & $0.751\pm0.02$ & $0.748\pm0.02$ \\
Loan & $0.658\pm0.08$ & $0.61\pm0.1$ & $0.628\pm0.1$ & $0.647\pm0.08$ & $0.616\pm0.09$ & $0.677\pm0.09$ \\
Diabetes & $0.791\pm0.02$ & $0.781\pm0.02$ & $0.787\pm0.02$ & $0.789\pm0.02$ & $0.788\pm0.02$ & $0.787\pm0.02$ \\
German Credit & $0.667\pm0.05$ & $0.649\pm0.05$ & $0.659\pm0.06$ & $0.665\pm0.06$ & $0.66\pm0.06$ & $0.667\pm0.05$ \\
Bank Marketing & $0.847\pm0.02$ & $0.832\pm0.03$ & $0.839\pm0.03$ & $0.844\pm0.02$ & $0.838\pm0.02$ & $0.851\pm0.02$ \\




In [30]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auprc_values = []
            std_auprc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                    mean_auprc_values.append(np.round(mean_auprc, 3))

                    std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                    std_auprc_values.append(np.round(std_auprc, 2))
                except IndexError:
                    mean_auprc_values.append(0)
                    std_auprc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ \
& ${mean_auprc_values[5]}\\pm{std_auprc_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.827\pm0.02$ & $0.809\pm0.02$ & $0.824\pm0.02$ & $0.825\pm0.02$ & $0.819\pm0.02$ & $0.83\pm0.02$ & \\
Income & $0.789\pm0.02$ & $0.765\pm0.02$ & $0.781\pm0.02$ & $0.786\pm0.02$ & $0.786\pm0.02$ & $0.786\pm0.02$ & \\
Breast Cancer & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.993\pm0.0$ & \\
HR Analytic & $0.455\pm0.03$ & $0.449\pm0.03$ & $0.454\pm0.03$ & $0.449\pm0.04$ & $0.452\pm0.03$ & $0.452\pm0.03$ & \\
Loan & $0.795\pm0.05$ & $0.776\pm0.06$ & $0.78\pm0.06$ & $0.793\pm0.05$ & $0.78\pm0.06$ & $0.804\pm0.06$ & \\
Diabetes & $0.371\pm0.04$ & $0.357\pm0.04$ & $0.367\pm0.04$ & $0.366\pm0.04$ & $0.365\pm0.04$ & $0.366\pm0.04$ & \\
German Credit & $0.461\pm0.06$ & $0.443\pm0.06$ & $0.452\pm0.06$ & $0.456\pm0.07$ & $0.45\pm0.06$ & $0.458\pm0.06$ & \\
Bank Marketing & $0.466\pm0.05$ & $0.446\pm0.06$ & $0.455\pm0.05$ & $0.466\pm0.05$ & $0.453\pm0.05$ & $0.469\pm0.05$ & \\




In [31]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.871,0.010,0.827,0.017,less_positive_class,0.1,0.00,0.000
1,KMM,folktables_employment,0.857,0.014,0.809,0.022,less_positive_class,0.1,0.00,0.000
2,PSA,folktables_employment,0.867,0.011,0.824,0.017,less_positive_class,0.1,0.04,0.280
3,MRS,folktables_employment,0.869,0.011,0.825,0.017,less_positive_class,0.1,354.00,74.800
4,FW-MRS,folktables_employment,0.864,0.012,0.819,0.019,less_positive_class,0.1,397.50,54.491
5,FW-MRS$_{Neg}$,folktables_employment,0.874,0.010,0.830,0.016,less_positive_class,0.1,350.10,77.592
6,FW-MRS$_{SVM}$,folktables_employment,0.834,0.013,0.777,0.023,less_positive_class,0.1,271.80,31.044
7,Uniform,folktables_income,0.838,0.013,0.789,0.020,less_positive_class,0.1,0.00,0.000
8,KMM,folktables_income,0.820,0.014,0.765,0.020,less_positive_class,0.1,0.00,0.000
9,PSA,folktables_income,0.831,0.014,0.781,0.022,less_positive_class,0.1,0.16,0.703


In [32]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.round(3).groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS,4.3125,4.5000
FW-MRS$_{Neg}$,2.9375,2.7500
FW-MRS$_{SVM}$,6.0000,6.2500
KMM,5.7500,5.7500
MRS,3.0000,3.2500
PSA,4.2500,3.6875
Uniform,1.7500,1.8125
